In [1]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# load data
X =np.load('data/f7/initial_inputs.npy')
y=np.load('data/f7/initial_outputs.npy')

In [3]:
# Data preparation: Normalization
y_mean_val = np.mean(y)
y_std_val  = np.std(y)
y_norm    = (y - y_mean_val) / y_std_val if y_std_val > 0 else y

In [4]:
# The best
best_f = np.max(y_norm)
best_idx = np.argmax(y_norm)
print(f"The best data is: X = {X[best_idx]}, y_norm = {best_f:.4f}")

The best data is: X = [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984], y_norm = 3.7910


In [5]:
# The GP model
dim=6  # kernel n (dim) dimension
noise_assumption = 1e-6
rbf_lengthscale  = np.ones(dim) * 1
kernel = RBF(length_scale=1, length_scale_bounds='fixed')
model  = GaussianProcessRegressor(kernel=kernel, alpha=noise_assumption)
model.fit(X, y_norm)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP.If `None` is passed,the kernel `ConstantKernel() * RBF()` is used as default.Note thatthe kernel hyperparameters are optimized during fittingunless the bounds are marked as `""fixed""`or the argument `optimizer` is set to `None`.",RBF(length_scale=1)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-06
,kernel__length_scale,1
,kernel__length_scale_bounds,'fixed'
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",0
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple fu

In [6]:
# Grid for prediction
from scipy.stats import qmc

n_grid = 5000 
sampler = qmc.LatinHypercube(d=dim, seed=42)
sample = sampler.random(n=n_grid)

# compute bounds
l_bounds = X.min(axis=0)  
u_bounds = X.max(axis=0)  
x_grid_d = qmc.scale(sample, l_bounds, u_bounds)

In [7]:
# GP prediction ---
post_mean, post_std = model.predict(x_grid_d, return_std=True)
#print(post_mean)
#print(post_std)

In [8]:
#EI (acquisition function definition)
def expected_improvement(mean, std, best_f, xi=0.01):
    Z  = (mean - best_f - xi) / (std + 1e-9)
    ei = (mean - best_f - xi) * norm.cdf(Z) + std * norm.pdf(Z)
    ei[std <= 0.0] = 0.0
    return ei

ei_values = expected_improvement(post_mean, post_std, best_f, xi=0.01)
#print (ei_values)

In [9]:
# Calculate the maximum value
idx_next  = np.argmax(ei_values)# simple grid
x_next    = x_grid_d[idx_next]
std_next = post_std[idx_next]
y_next = post_mean[idx_next]

print(f"proposed query (EI): x1={x_next[0]:.8f}, x2={x_next[1]:.8f}, x3={x_next[2]:.8f}, x4={x_next[3]:.8f}, x5={x_next[4]:.8f}, x6={x_next[5]:.8f} ")
print(f"proposed query std (EI): {std_next:.6f}")
print(f"proposed query y_next (EI): {y_next:.6f}")

proposed query (EI): x1=0.09557010, x2=0.09963534, x3=0.25493264, x4=0.25910931, x5=0.11618892, x6=0.94916761 
proposed query std (EI): 0.233255
proposed query y_next (EI): 5.952995


In [10]:
# Matern Kernel

In [11]:
from sklearn.gaussian_process.kernels import Matern

# Kernel Matern 5/2 with ARD
kernel = Matern(
    length_scale=np.ones(dim),
    length_scale_bounds=(1e-3, 1e3),
    nu=2.5
)#search range
model  = GaussianProcessRegressor(kernel=kernel,
    alpha=1e-6,
    n_restarts_optimizer=5)
model.fit(X, y_norm)

# length-scales
ls = model.kernel_.length_scale
print(f"learned Length-scales: x1={ls[0]:.4f}, x2={ls[1]:.4f}, x3={ls[2]:.4f}, x4={ls[3]:.4f}, x5={ls[4]:.4f}, x6={ls[5]:.4f}")

learned Length-scales: x1=587.1013, x2=84.9702, x3=1.5869, x4=0.2268, x5=48.7616, x6=0.0133


In [12]:
# Grid for prediction
from scipy.stats import qmc

n_grid = 5000 
sampler = qmc.LatinHypercube(d=dim, seed=42)
sample = sampler.random(n=n_grid)

# update bounds
l_bounds = X.min(axis=0)  
u_bounds = X.max(axis=0)  
x_grid_d = qmc.scale(sample, l_bounds, u_bounds)

# GP prediction ---
post_mean, post_std = model.predict(x_grid_d, return_std=True)
#print(post_mean)
#print(post_std)

# EI
def expected_improvement(mean, std, best_f, xi=0.01):
    Z  = (mean - best_f - xi) / (std + 1e-9)
    ei = (mean - best_f - xi) * norm.cdf(Z) + std * norm.pdf(Z)
    ei[std <= 0.0] = 0.0
    return ei

ei_values = expected_improvement(post_mean, post_std, best_f, xi=0.01)
#print (ei_values)
# The next point
idx_next  = np.argmax(ei_values)# simple grid
x_next    = x_grid_d[idx_next]
print(f"Mejor punto sugerido (EI): x1={x_next[0]:.8f}, x2={x_next[1]:.8f}, x3={x_next[2]:.8f}, x4={x_next[3]:.8f}, x5={x_next[4]:.8f}, x6={x_next[5]:.8f} ")

Mejor punto sugerido (EI): x1=0.63074613, x2=0.58377760, x3=0.24139399, x4=0.18943042, x5=0.55017795, x6=0.73638024 


In [13]:
# Write output file
import os
folder = 'results'
filename = 'output_F7.txt'
file_path = os.path.join(folder, filename)
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(f"{x_next[0]:.6f} {x_next[1]:.6f} {x_next[2]:.6f} {x_next[3]:.6f} {x_next[4]:.6f} {x_next[5]:.6f}\n")
